# Train the drone delivery agents on Google Colab

This notebook runs the same training as the repository's scripts: 10 hyperparameter
configurations each for **DQN, PPO, A2C and REINFORCE**, 600,000 environment steps per run,
every run evaluated on 20 shifts it never trained on. It then writes the comparison plots and
tables and packs the trained models and logs for download.

**Runtime:** a CPU runtime is enough. The networks are small and the time goes into stepping
the simulation, so a GPU does not make it faster. Each run takes about 6–7 minutes on one core;
runs go in parallel across the cores Colab gives you (usually 2 on the free tier, so the full
40 runs take roughly 2–2.5 hours). Set `FRAMES` below for a quick check first.

It trains the code on GitHub, so push your latest changes before running it.

In [ ]:
# ---- settings ----
REPO = "https://github.com/supserrr/shima_serein_rl_summative.git"
BRANCH = "main"
ALGOS = ["dqn", "ppo", "a2c", "reinforce"]  # train any subset
FRAMES = None   # None = the full 600,000 steps per run; e.g. 50_000 for a quick check
RUNS = None     # None = all 10 configurations; e.g. [0, 3] for just those two

import os
JOBS = max(1, os.cpu_count() or 1)
print(f"{JOBS} CPU cores: {JOBS} runs at a time")

## 1. Get the code and install the pinned dependencies

In [ ]:
%cd /content
!rm -rf shima_serein_rl_summative
!git clone --depth 1 --branch {BRANCH} {REPO}
%cd /content/shima_serein_rl_summative
# the versions in uv.lock; CPU-only PyTorch keeps the download small
!pip install -q torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q torchrl==0.15.0 tensordict==0.15.6 gymnasium==1.4.0 "numpy>=2.5.5" "pandas>=3.1.1" "matplotlib>=3.11.2" pytest

## 2. Check the environment

In [ ]:
!python -m pytest -q

## 3. Train

Each algorithm runs its sweep, writes `logs/<algo>/sweep_summary.csv` and copies its best run to
`models/…/<algo>_best`. The line printed for each run is its mean return on the 20 validation shifts.

In [ ]:
import subprocess, sys, time

def sweep(algo):
    cmd = [sys.executable, "-m", "training.dqn_training"] if algo == "dqn" else \
          [sys.executable, "-m", "training.pg_training", "--algo", algo]
    cmd += ["--runs", *map(str, RUNS)] if RUNS else ["--sweep"]  # --sweep means all 10
    cmd += ["--jobs", str(JOBS)]
    if FRAMES:
        cmd += ["--frames", str(FRAMES)]
    t0 = time.time()
    with open(f"logs/sweep_{algo}.out", "w") as log, subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            log.write(line)
            if line.startswith(f"[{algo}]"):  # one line per finished run; the rest goes to the log file
                print(line, end="", flush=True)
    print(f"{algo}: done in {(time.time() - t0) / 60:.0f} min (exit {p.returncode})\n")

for algo in ALGOS:
    sweep(algo)

## 4. Compare the agents

In [ ]:
!python -m training.analyze --retest   # recompute the baselines, test evaluation and seed check
from IPython.display import Markdown, Image, display
display(Markdown(open("assets/tables/headline.md").read()))

In [ ]:
import glob
for png in sorted(glob.glob("assets/plots/*.png")):
    print(png)
    display(Image(png, width=760))

## 5. Download the results

The zip holds `models/`, `logs/` and `assets/`. Unzip it over your local copy of the repository,
then `uv run main.py` plays the new best agent in the 3D viewer.

In [ ]:
!zip -qr /content/drone-rl-results.zip models logs assets
from google.colab import files
files.download("/content/drone-rl-results.zip")

In [ ]:
# Optional: keep a copy in Google Drive instead (asks for access to your Drive)
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drone-rl-results.zip /content/drive/MyDrive/